# 👁️ Computer Vision — CNN, OCR e Detecção de Objetos
**FIAP Postech — IA para Devs | Fase 1 · Computer Vision**

---

## O que você vai aprender
1. Fundamentos de Visão Computacional — como o computador "vê" imagens
2. **CNN** (Redes Neurais Convolucionais) — arquitetura e treinamento
3. **Transfer Learning** com redes pré-treinadas (ResNet, MobileNet)
4. **OCR** — extraindo texto de imagens
5. Detecção de objetos com **YOLO** (introdução)
6. Visão geral de GANs

---
> **Por que Computer Vision importa?**  
> Diagnóstico médico por imagem, carros autônomos, reconhecimento facial, inspeção industrial, leitura de documentos — tudo usa CV.

---
## 0. Instalação

In [ ]:
!pip install tensorflow opencv-python-headless pillow pytesseract matplotlib --quiet

# Para OCR com Tesseract (Linux/Colab):
# !apt-get install -y tesseract-ocr tesseract-ocr-por
# Para Windows: baixe em https://github.com/UB-Mannheim/tesseract/wiki

print('✅ Instalação concluída')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import warnings
warnings.filterwarnings('ignore')

import cv2
from PIL import Image, ImageDraw, ImageFont

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.preprocessing.image import img_to_array, array_to_img

print(f'✅ TensorFlow {tf.__version__}')
print(f'   GPU disponível: {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 1. Como o Computador Vê Imagens

In [ ]:
# Carregando MNIST — dígitos escritos à mão (28x28 pixels, grayscale)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f'Train: {X_train.shape}  →  {X_train.shape[0]} imagens de {X_train.shape[1]}x{X_train.shape[2]} pixels')
print(f'Test:  {X_test.shape}')
print(f'\nValores de pixel: min={X_train.min()}, max={X_train.max()}')
print(f'Rótulos: {np.unique(y_train)}  (dígitos 0-9)')

In [ ]:
# Visualizando imagens e seus valores numéricos
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i in range(5):
    # Linha 1: imagem
    axes[0, i].imshow(X_train[i], cmap='gray')
    axes[0, i].set_title(f'Label: {y_train[i]}', fontweight='bold')
    axes[0, i].axis('off')

    # Linha 2: heatmap dos valores numéricos
    im = axes[1, i].imshow(X_train[i], cmap='Blues')
    axes[1, i].set_title(f'Pixels (0-255)', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle('Uma imagem para o computador = matriz de números', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nUm único dígito "3" como matriz 28x28 (trecho 10x10):')
idx_3 = np.where(y_train == 3)[0][0]
print(X_train[idx_3][8:18, 8:18])

In [ ]:
# Canais de cor — RGB
(X_c_train, y_c_train), (X_c_test, y_c_test) = cifar10.load_data()
classes_cifar = ['avião','carro','pássaro','gato','cervo','cachorro','sapo','cavalo','navio','caminhão']

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for i in range(5):
    img = X_c_train[i]  # 32x32x3
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'{classes_cifar[y_c_train[i][0]]}', fontweight='bold')
    axes[0, i].axis('off')

# Canal R separado
for i in range(5):
    img = X_c_train[i]
    axes[1, i].imshow(img[:,:,0], cmap='Reds')  # apenas canal vermelho
    axes[1, i].set_title('Canal R', fontsize=9)
    axes[1, i].axis('off')

plt.suptitle(f'CIFAR-10: imagens 32×32×3 (RGB) | Shape={X_c_train.shape}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. Pré-processamento de Imagens

In [ ]:
# Normalização dos pixels: 0-255 → 0.0-1.0
X_train_n = X_train.astype('float32') / 255.0
X_test_n  = X_test.astype('float32') / 255.0

# Reshape: CNN espera (batch, height, width, channels)
X_train_cnn = X_train_n.reshape(-1, 28, 28, 1)  # grayscale = 1 canal
X_test_cnn  = X_test_n.reshape(-1, 28, 28, 1)

# One-hot encoding dos labels: 3 → [0,0,0,1,0,0,0,0,0,0]
y_train_oh = keras.utils.to_categorical(y_train, 10)
y_test_oh  = keras.utils.to_categorical(y_test, 10)

print(f'X_train_cnn shape: {X_train_cnn.shape}')
print(f'y_train_oh shape:  {y_train_oh.shape}')
print(f'\nExemplo label: {y_train[0]} → one-hot: {y_train_oh[0]}')

---
## 3. Redes Neurais Convolucionais (CNN)

### Arquitetura típica:
```
Input → [Conv2D → ReLU → MaxPool] × N → Flatten → Dense → Softmax → Output
```

| Camada | O que faz |
|---|---|
| **Conv2D** | Aplica filtros que detectam bordas, formas, texturas |
| **ReLU** | Ativação não-linear: f(x) = max(0, x) |
| **MaxPool** | Reduz dimensionalidade mantendo as features mais fortes |
| **Flatten** | Transforma matriz 2D em vetor 1D |
| **Dense** | Camada totalmente conectada (MLP clássico) |
| **Softmax** | Converte saídas em probabilidades (soma = 1) |

In [ ]:
# Construindo a CNN
def criar_cnn_mnist():
    model = keras.Sequential([
        # Bloco 1: detecta features básicas (bordas, linhas)
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='same'),
        layers.MaxPooling2D((2, 2)),

        # Bloco 2: detecta features mais complexas (curvas, formas)
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # Bloco 3
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),

        # Classificador
        layers.Flatten(),
        layers.Dropout(0.5),          # regularização: desliga 50% dos neurônios no treino
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')  # 10 classes
    ])
    return model

cnn = criar_cnn_mnist()
cnn.summary()

In [ ]:
# Compilando
cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Treinando — use epochs=5 para ser rápido; aumente para melhor resultado
history = cnn.fit(
    X_train_cnn, y_train_oh,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Avaliação
loss, acc = cnn.evaluate(X_test_cnn, y_test_oh, verbose=0)
print(f'\n🎯 Test Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'   Test Loss: {loss:.4f}')

# Curvas de aprendizado
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'], 'b-o', label='Train')
axes[0].plot(history.history['val_accuracy'], 'r-o', label='Validation')
axes[0].set_title('Accuracy por Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], 'b-o', label='Train')
axes[1].plot(history.history['val_loss'], 'r-o', label='Validation')
axes[1].set_title('Loss por Epoch', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.suptitle('Curvas de Aprendizado — CNN MNIST', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualizando predições
y_pred_proba = cnn.predict(X_test_cnn[:25])
y_pred_class = np.argmax(y_pred_proba, axis=1)

fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_test[i], cmap='gray')
    real, pred = y_test[i], y_pred_class[i]
    cor = '#2ecc71' if real == pred else '#e74c3c'
    ax.set_title(f'R:{real} P:{pred}', color=cor, fontweight='bold', fontsize=10)
    ax.axis('off')

plt.suptitle('Predições CNN — Verde=Correto, Vermelho=Errado', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Transfer Learning — Aproveitando Redes Pré-Treinadas

Treinar uma CNN do zero exige milhões de imagens e GPUs. **Transfer learning** usa o conhecimento de redes já treinadas (ImageNet, 1M imagens) e só retreina as últimas camadas para o seu problema.

In [ ]:
# MobileNetV2 pré-treinada no ImageNet
base_model = MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,    # remove a camada de classificação original
    weights='imagenet'    # pesos pré-treinados
)

# Congelando os pesos da base — não treina as conv layers
base_model.trainable = False

print(f'Total de parâmetros: {base_model.count_params():,}')
print(f'Parâmetros treináveis: 0 (base congelada)')

In [ ]:
# Adicionando cabeça de classificação para CIFAR-10
inputs = keras.Input(shape=(32, 32, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)  # 10 classes CIFAR

transfer_model = keras.Model(inputs, outputs)

transfer_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('Parâmetros treináveis:', transfer_model.trainable_variables.__len__())
transfer_model.summary()

In [ ]:
# Pré-processamento CIFAR
X_c_train_n = X_c_train.astype('float32') / 255.0
X_c_test_n  = X_c_test.astype('float32') / 255.0

# Treinando só as últimas camadas (rápido!)
history_tl = transfer_model.fit(
    X_c_train_n, y_c_train,
    epochs=3,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

loss_tl, acc_tl = transfer_model.evaluate(X_c_test_n, y_c_test, verbose=0)
print(f'\n🎯 Transfer Learning — Test Accuracy: {acc_tl:.4f} ({acc_tl*100:.1f}%)')
print(f'   (Apenas 3 epochs, sem treinar as camadas convolucionais!)')

---
## 5. OCR — Extraindo Texto de Imagens

In [ ]:
# Criando uma imagem com texto para testar OCR
from PIL import Image, ImageDraw, ImageFont
import io

img_pil = Image.new('RGB', (400, 120), color='white')
draw = ImageDraw.Draw(img_pil)
draw.text((20, 20), 'FIAP Postech', fill='black')
draw.text((20, 50), 'IA para Devs - 2025', fill='black')
draw.text((20, 80), 'Machine Learning Module', fill='darkblue')

plt.figure(figsize=(7, 2.5))
plt.imshow(img_pil)
plt.axis('off')
plt.title('Imagem gerada para OCR', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# OCR com pytesseract
try:
    import pytesseract

    # Para Windows: descomente e ajuste o caminho
    # pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

    texto = pytesseract.image_to_string(img_pil)
    print('📝 Texto extraído pelo OCR:')
    print(texto)

    # Dados detalhados (posição de cada palavra)
    dados = pytesseract.image_to_data(img_pil, output_type=pytesseract.Output.DICT)
    print('\n📊 Palavras e bounding boxes:')
    for i, palavra in enumerate(dados['text']):
        if palavra.strip():
            conf = dados['conf'][i]
            x, y, w, h = dados['left'][i], dados['top'][i], dados['width'][i], dados['height'][i]
            print(f'  "{palavra}" — confiança: {conf}% | pos: ({x},{y}) size: {w}x{h}')

except Exception as e:
    print(f'⚠️ Tesseract não instalado: {e}')
    print('\nPara instalar:')
    print('  Linux/Colab: !apt-get install tesseract-ocr')
    print('  Windows: https://github.com/UB-Mannheim/tesseract/wiki')
    print('  Mac: brew install tesseract')
    print('\n💡 Alternativa cloud: Google Vision API, AWS Textract, Azure Computer Vision')

In [ ]:
# Pré-processamento de imagem para melhorar OCR
# OpenCV transforma imagem para facilitar a leitura

img_array = np.array(img_pil)

# Convertendo para grayscale
gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)

# Binarização (threshold)
_, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# Denoising
denoised = cv2.fastNlMeansDenoising(gray, h=10)

fig, axes = plt.subplots(1, 3, figsize=(14, 3))
axes[0].imshow(gray, cmap='gray'); axes[0].set_title('Grayscale', fontweight='bold'); axes[0].axis('off')
axes[1].imshow(binary, cmap='gray'); axes[1].set_title('Binarizado', fontweight='bold'); axes[1].axis('off')
axes[2].imshow(denoised, cmap='gray'); axes[2].set_title('Denoised', fontweight='bold'); axes[2].axis('off')

plt.suptitle('Pré-processamento para OCR', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Imagens binarizadas geralmente dão melhores resultados no OCR')

---
## 6. Detecção de Objetos — Conceitos YOLO

YOLO (You Only Look Once) processa a imagem inteira de uma vez e prediz bounding boxes + classes simultâneamente.

In [ ]:
# Simulando uma detecção de objetos com bounding boxes
# (sem rodar YOLO real para não exigir GPU)

img_det = np.ones((480, 640, 3), dtype=np.uint8) * 240  # imagem cinza

# Desenhando objetos simulados
cv2.rectangle(img_det, (50, 80), (220, 300), (200, 100, 50), -1)   # "pessoa"
cv2.rectangle(img_det, (280, 150), (580, 380), (50, 150, 200), -1)  # "carro"
cv2.circle(img_det, (520, 100), 50, (80, 200, 80), -1)             # "bola"

# Bounding boxes simulando saída YOLO
deteccoes = [
    {'label': 'pessoa', 'conf': 0.94, 'box': (50, 80, 220, 300), 'cor': (255, 80, 0)},
    {'label': 'carro',  'conf': 0.87, 'box': (280, 150, 580, 380), 'cor': (0, 100, 255)},
    {'label': 'bola',   'conf': 0.72, 'box': (470, 50, 570, 150), 'cor': (0, 200, 0)},
]

img_viz = img_det.copy()
for d in deteccoes:
    x1, y1, x2, y2 = d['box']
    cv2.rectangle(img_viz, (x1, y1), (x2, y2), d['cor'], 3)
    label = f"{d['label']} {d['conf']:.0%}"
    cv2.rectangle(img_viz, (x1, y1-30), (x1+len(label)*11, y1), d['cor'], -1)
    cv2.putText(img_viz, label, (x1+2, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(img_viz, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Simulação de Detecção de Objetos (estilo YOLO output)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 YOLO detecta:')
for d in deteccoes:
    x1,y1,x2,y2 = d['box']
    print(f"  {d['label']}: confiança={d['conf']:.0%} | bbox=({x1},{y1}) → ({x2},{y2})")

In [ ]:
# Como usar YOLO de verdade (requer ultralytics)
print('=' * 55)
print('  Como usar YOLOv8 na prática')
print('=' * 55)
print('''
# 1. Instalar
pip install ultralytics

# 2. Detecção em imagem
from ultralytics import YOLO

model = YOLO('yolov8n.pt')          # baixa modelo pré-treinado
results = model('minha_imagem.jpg') # roda detecção
results[0].show()                   # mostra resultado com bboxes

# 3. Resultados detalhados
for r in results:
    boxes = r.boxes          # bounding boxes
    masks = r.masks          # segmentation masks
    probs = r.probs          # class probabilities

# 4. Em vídeo (webcam)
results = model(source=0, show=True)  # 0 = webcam

# 5. Fine-tuning no seu dataset
model.train(data='meu_dataset.yaml', epochs=50)
''')

---
## 7. GAN — Visão Geral

**Generative Adversarial Network** = dois modelos competindo:

```
Generator (G)     →  cria imagens falsas
Discriminator (D) →  tenta distinguir real vs falso

G tenta enganar D.
D tenta não ser enganado.
Resultado: G aprende a criar imagens realistas.
```

In [ ]:
# Arquitetura simplificada de uma GAN para MNIST

LATENT_DIM = 100  # dimensão do ruído de entrada do Generator

# Generator: ruído aleatório → imagem 28x28
generator = keras.Sequential([
    layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(LATENT_DIM,)),
    layers.BatchNormalization(),
    layers.LeakyReLU(0.2),
    layers.Reshape((7, 7, 256)),
    layers.Conv2DTranspose(128, (5,5), strides=(1,1), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(0.2),
    layers.Conv2DTranspose(64, (5,5), strides=(2,2), padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.LeakyReLU(0.2),
    layers.Conv2DTranspose(1, (5,5), strides=(2,2), padding='same', use_bias=False, activation='tanh')
], name='Generator')

# Discriminator: imagem 28x28 → real ou falso
discriminator = keras.Sequential([
    layers.Conv2D(64, (5,5), strides=(2,2), padding='same', input_shape=(28,28,1)),
    layers.LeakyReLU(0.2),
    layers.Dropout(0.3),
    layers.Conv2D(128, (5,5), strides=(2,2), padding='same'),
    layers.LeakyReLU(0.2),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid')
], name='Discriminator')

print('Generator:')
generator.summary()
print('\nDiscriminator:')
discriminator.summary()

In [ ]:
# Visualizando imagens geradas pelo Generator (não treinado — ruído puro)
ruido = tf.random.normal([16, LATENT_DIM])
imgs_geradas = generator(ruido, training=False)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(imgs_geradas[i, :, :, 0], cmap='gray')
    ax.axis('off')

plt.suptitle('Generator antes do treino (ruído puro)\nApós treinar → dígitos realistas!',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Após ~100 epochs de treino, o Generator produz dígitos indistinguíveis dos reais.')
print('   GANs são a base de: deepfakes, geração de arte, aumento de dados, super-resolução.')

---
## 8. Mapa Mental — Computer Vision na Fase 1

```
Computer Vision
│
├── Representação
│   ├── Grayscale: matriz 2D (H × W)
│   └── RGB: tensor 3D (H × W × 3)
│
├── Pré-processamento
│   ├── Normalização (÷255)
│   ├── Resize
│   ├── Augmentation (flip, crop, rotate)
│   └── Binarização (para OCR)
│
├── CNN
│   ├── Conv2D → detecta features locais
│   ├── MaxPool → reduz dimensão
│   ├── Dense → classifica
│   └── Transfer Learning → reutiliza pesos
│
├── OCR
│   ├── Tesseract (local)
│   └── Cloud: Google Vision, AWS Textract, Azure
│
├── Detecção de Objetos
│   └── YOLO → bounding boxes + classes em tempo real
│
└── GAN
    ├── Generator → cria imagens
    └── Discriminator → distingue real/falso
```

---
*Notebook criado para FIAP Postech — IA para Devs | Fase 1 · Computer Vision*